In [1]:
import pandas as pd
import numpy as np

In [ ]:
# Cargar datos
transactions = pd.read_parquet('../transactions_agg.parquet') #DATA SIN DUPLICADOS


In [ ]:
# Anonimizamos price
transactions['price'] = transactions['price'] * 50000

In [ ]:
# Calculo de metiras RFM
rfm = transactions.groupby('customer_id').agg({
    'price': ['sum', 'count'],  # Monetary y Frequency
    't_dat': 'max'              # Para  recency
}).reset_index()

In [5]:
rfm.columns = ['customer_id', 'monetary', 'frequency', 'last_purchase']

In [6]:
# Recency
rfm['recency'] = (pd.to_datetime('2020-09-30') - pd.to_datetime(rfm['last_purchase'])).dt.days

In [7]:
# CLASIFICACIÓN ABC (Monetary)
rfm = rfm.sort_values('monetary', ascending=False)
rfm['cumulative_percent'] = rfm['monetary'].cumsum() / rfm['monetary'].sum() * 100


In [9]:
# Función vectorizada para ABC
conditions_abc = [
    (rfm['cumulative_percent'] <= 80),
    (rfm['cumulative_percent'] >80),
    (rfm['cumulative_percent'] > 95)
]
choices_abc = ['A', 'B', 'C']
rfm['abc_class'] = np.select(conditions_abc, choices_abc,default='C')

In [ ]:
# Clasificación XYZ variabilidad
# Calcular coeficiente de variación por cliente
customer_stats = transactions.groupby('customer_id').agg({
    'price': ['mean', 'std']
}).reset_index()
customer_stats.columns = ['customer_id', 'avg_price', 'std_price']

In [11]:
# Evitar división por cero
customer_stats['cv'] = customer_stats['std_price'] / customer_stats['avg_price'].replace(0, np.nan)

In [13]:
# Clasificación XYZ
conditions_xyz = [
    (customer_stats['cv'] <= 0.5),
    (customer_stats['cv'] <= 1.0),
    (customer_stats['cv'] > 1.0)
]
choices_xyz = ['X', 'Y', 'Z']
customer_stats['xyz_class'] = np.select(conditions_xyz, choices_xyz,default='Z')

In [14]:
# Combinar ABC y XYZ
abc_xyz = rfm[['customer_id', 'abc_class']].merge(
    customer_stats[['customer_id', 'xyz_class']], 
    on='customer_id', 
    how='left'
)
abc_xyz['abc_xyz_segment'] = abc_xyz['abc_class'] + abc_xyz['xyz_class']

In [18]:
#CONSOLIDAR
final_df = rfm[['customer_id', 'monetary', 'frequency', 'recency', 'abc_class']].merge(
    customer_stats[['customer_id', 'xyz_class']], 
    on='customer_id',
    how='left'
)

In [19]:
# Crear columna combinada
final_df['abc_xyz_segment'] = final_df['abc_class'] + final_df['xyz_class']

In [ ]:
print(final_df.head())

                                         customer_id      monetary  frequency  \
0  be1981ab818cf4ef6765b2ecaea7a2cbf14ccd6e8a7ee9...  2.883820e+06       1636   
1  a65f77281a528bf5c1e9f270141d601d116e1df33bf9df...  2.546059e+06       1302   
2  03d0011487606c37c1b1ed147fc72f285a50c05f00b971...  2.498358e+06        855   
3  191071b0e1f2e94a557f1a0b4cea3de55faf1581b1f464...  2.384101e+06        670   
4  b4db5e5259234574edfff958e170fe3a5e13b6f146752c...  2.383100e+06       1310   

   recency abc_class xyz_class abc_xyz_segment  
0       15         A         Y              AY  
1       10         A         Y              AY  
2       14         A         Y              AY  
3       11         A         Y              AY  
4        8         A         Y              AY  


In [ ]:
#Vista final con segmentos abc_xyz
final_df

,customer_id,monetary,frequency,recency,abc_class,xyz_class,abc_xyz_segment
0,be1981ab818cf4ef6765b2ecaea7a2cbf14ccd6e8a7ee9...,2.883820e+06,1636,15,A,Y,AY
1,a65f77281a528bf5c1e9f270141d601d116e1df33bf9df...,2.546059e+06,1302,10,A,Y,AY
2,03d0011487606c37c1b1ed147fc72f285a50c05f00b971...,2.498358e+06,855,14,A,Y,AY
3,191071b0e1f2e94a557f1a0b4cea3de55faf1581b1f464...,2.384101e+06,670,11,A,Y,AY
4,b4db5e5259234574edfff958e170fe3a5e13b6f146752c...,2.383100e+06,1310,8,A,Y,AY
...,...,...,...,...,...,...,...
1362276,b6345f8bf3fdb4a9833a812e11aaebfd6e2f256e8834d4...,4.237288e+01,1,52,B,Z,BZ
1362277,a2efffd45ccdc46ea647c1e18037200934b7c250a5f211...,4.237288e+01,1,202,B,Z,BZ
1362278,ed43af17bc5ec016ea75b7cd5ab0948f526ff94a5776a7...,4.237288e+01,1,713,B,Z,BZ
1362279,7ba686bcc845f1ea49b00e32c98cf6d9132fc29395a580...,4.237288e+01,1,455,B,Z,BZ


In [ ]:
# Guardar resultado
final_df.to_parquet("lakehouse/abc_xyz_segment.parquet", engine='pyarrow', index=False)